Requires **flowmeter** — install once from the repo root:

```bash
pip install -e tools/flowmeter/
```

# Imports

In [ ]:
import os
import glob
import tempfile

import pandas as pd
from flowmeter import convert_pcap_to_csv

# Inputs

In [ ]:
DATA_ROOT = "../data"

# Preprocessing

In [ ]:
def find_pcap_folders(root_dir):
    """Return every folder under root_dir that contains at least one PCAP file."""
    found = []
    for dirpath, _, filenames in os.walk(root_dir):
        if any(f.endswith((".pcap", ".pcapng")) for f in filenames):
            found.append(dirpath)
    return sorted(found)


def generate_merged_csv_from_pcaps(pcap_dir, output_filename):
    """
    Convert every PCAP/pcapng in pcap_dir to flow features and write a
    single merged CSV into that same folder.

    Parameters
    ----------
    pcap_dir : str
        Folder containing .pcap / .pcapng files. The CSV is written here.
    output_filename : str
        Name of the output CSV file (e.g. 'merged_benign.csv').

    Returns
    -------
    str
        Full path to the merged CSV file.
    """
    pcap_files = sorted(
        glob.glob(os.path.join(pcap_dir, "*.pcap")) +
        glob.glob(os.path.join(pcap_dir, "*.pcapng"))
    )

    if not pcap_files:
        raise FileNotFoundError(f"No PCAP files found in: {pcap_dir}")

    print(f"[{os.path.basename(pcap_dir)}] {len(pcap_files)} PCAP file(s)")

    frames = []
    with tempfile.TemporaryDirectory() as tmp_dir:
        for i, pcap_path in enumerate(pcap_files, 1):
            pcap_name = os.path.basename(pcap_path)
            print(f"  [{i}/{len(pcap_files)}] {pcap_name}")

            tmp_csv = os.path.join(tmp_dir, f"{pcap_name}.csv")
            n_flows = convert_pcap_to_csv(pcap_path, tmp_csv)
            print(f"    → {n_flows} flows")

            if n_flows > 0:
                frames.append(pd.read_csv(tmp_csv))

    if not frames:
        raise RuntimeError(f"No flows extracted from: {pcap_dir}")

    merged_df = pd.concat(frames, ignore_index=True)
    merged_df["label"] = os.path.basename(pcap_dir)

    output_path = os.path.join(pcap_dir, output_filename)
    merged_df.to_csv(output_path, index=False)

    print(f"  ✓ {output_path}  ({len(merged_df)} flows)\n")
    return output_path

In [ ]:
pcap_folders = find_pcap_folders(DATA_ROOT)
print(f"Found {len(pcap_folders)} folder(s) with PCAPs\n")

for folder_path in pcap_folders:
    folder_name = os.path.basename(folder_path)
    generate_merged_csv_from_pcaps(folder_path, f"merged_{folder_name}.csv")